In [1]:
import math

import tensorflow as tf
import keras_cv
from tensorflow import keras

ERROR:absl:cannot import name 'runtime_version' from 'google.protobuf' (d:\Users\Saulete\Downloads\venv\Lib\site-packages\google\protobuf\__init__.py)
Traceback (most recent call last):
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\rlds\__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\rlds\envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\core\__init__.py", line 21, in <module>
    from tensorflow_datasets.core import community
  File "d:\Users\Saulete\Downloads\venv

You do not have pycocotools installed, so KerasCV pycoco metrics are not available. Please run `pip install pycocotools`.
You do not have pyococotools installed, so the `PyCOCOCallback` API is not available.
You do not have Waymo Open Dataset installed, so KerasCV Waymo metrics are not available.


In [2]:
# load the pipeline, then get the diffusion/denoise mode

model = keras_cv.models.StableDiffusion(img_width=256, img_height=256)
diffusion_model = model.diffusion_model

By using this model checkpoint, you acknowledge that its usage is subject to the terms of the CreativeML Open RAIL-M license at https://raw.githubusercontent.com/CompVis/stable-diffusion/main/LICENSE


In [3]:
# find the op/layer that we can use to split the model into two roughly equal chunks

def find_split_layer(model):
    total_size = 0

    for layer in model.layers:
        if layer.weights:
            # print(layer.name)
            if (isinstance(layer.weights, list)):
                  for w in layer.weights:
                    # print(w.shape, w.dtype)
                    total_size = total_size + w.numpy().size
    # print("total size:", total_size)
    half_size = total_size / 2

    first_layers = []
    accumulator = 0 
    for layer in model.layers:
        first_layers.append(layer.name)
        # print(first_layers)
        if layer.weights:
            if (isinstance(layer.weights, list)):
                for w in layer.weights:
                    accumulator = accumulator + w.numpy().size
                if accumulator > half_size:
                    return first_layers, layer.name

In [4]:
# find the edges crossing both chunks
# use them as the output tensors of the first chunk and the input tensors of the second chunk

def find_boundary_tensors(model, first_layers, end_of_first_chunk):
    
    boundary_tensors = []
    boundary_input_layers = []
    in_second_chunk = False
    
    for l in model.layers:
        if in_second_chunk:
            #print(l.name)
            if (isinstance(l.input, list)):
                for i in l.input:
                    #print("  ", i.node.layer.name)
                    if (i.node.layer.name in first_layers):
                        #print("  ", i.node.layer.name)
                        #print(boundary_input_layers)
                        if (i.node.layer.name not in boundary_input_layers):
                            # print(boundary_tensors)
                            boundary_tensors.append(i)
                            boundary_input_layers.append(i.node.layer.name)
            else:
                # print("  whatever", l.input.node.layer.name)
                if (l.input.node.layer.name in first_layers):
                    # print("  yes:", l.input.layer.name)
                    boundary_tensors.append(l.input)
                    boundary_input_layers.append(i.input.name)
                    
        elif (l.name == end_of_first_chunk):
            in_second_chunk = True
            
    return boundary_tensors

In [5]:
first_layers, end_of_first_chunk = find_split_layer(diffusion_model)
boundary_tensors = find_boundary_tensors(diffusion_model, first_layers, end_of_first_chunk)

# construct the two chunks
first_part = keras.Model(diffusion_model.inputs, boundary_tensors)
second_part = keras.Model(boundary_tensors, diffusion_model.outputs)

In [6]:
prompt_1 = "A watercolor painting of a Golden Retriever at the beach"
encoding_1 = model.encode_text(prompt_1)

def get_timestep_embedding(timestep, batch_size, dim=320, max_period=10000):
    half = dim // 2
    freqs = tf.math.exp(
        -math.log(max_period) * tf.range(0, half, dtype=tf.float32) / half
    )

    args = tf.convert_to_tensor([timestep], dtype=tf.float32) * freqs
    embedding = tf.concat([tf.math.cos(args), tf.math.sin(args)], 0)
    embedding = tf.reshape(embedding, [1, -1])
    return tf.repeat(embedding, batch_size, axis=0)

In [7]:
def representative_data_gen_first():
    for i in range(100):
        em = get_timestep_embedding(i + 1, 1)        # (1, 320)
        noise = tf.random.normal((1, 32, 32, 4))
        ctx = encoding_1                             # (1, 77, 768)
        
        yield [noise, em, ctx]

In [ ]:
# when converting a Keras model to a tflite model, it's saved to a saved_model first
# in a saved_model, the 13 inputs are named args_0, args_0_1, args_0_2,..., args_0_12
def representative_data_gen_second():
    for i in range(100):
        em = get_timestep_embedding(i+1, 1) 
        noise = tf.random.normal((1, 32, 32, 4))
        
        # Get the 13 intermediate tensors from the first chunk
        a = first_part((noise, em, encoding_1))
        
        # Yield a dictionary mapping our NEW input names to these tensors
        yield {
            f'static_second_input_{j}': a[j] for j in range(len(a))
        }

In [10]:
static_input_17 = tf.keras.Input(batch_shape=(1, 320), dtype=tf.float32, name="timestep_embedding")
static_input_18 = tf.keras.Input(batch_shape=(1, 32, 32, 4), dtype=tf.float32, name="latent_noise")
static_input_16 = tf.keras.Input(batch_shape=(1, 77, 768), dtype=tf.float32, name="context_embedding")
static_inputs_list = [static_input_18, static_input_17, static_input_16]

static_first_part = tf.keras.Model(
    inputs=static_inputs_list,
    outputs=first_part(static_inputs_list)
)

converter1 = tf.lite.TFLiteConverter.from_keras_model(static_first_part)
converter1.optimizations = [tf.lite.Optimize.DEFAULT]
converter1.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter1.target_spec.supported_types = [tf.int8]
converter1.inference_input_type = tf.int8
converter1.inference_output_type = tf.int8

converter1.representative_dataset = representative_data_gen_first
first_chunk_qint8_tflite = converter1.convert()

with open('/tmp/diffusion_model_first_qint8.tflite', 'wb') as f:
        f.write(first_chunk_qint8_tflite)

INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmpg_rifxxi\assets


INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmpg_rifxxi\assets
d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow\lite\python\convert.py:789: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


In [ ]:
static_inputs_second_part = []

for i, tensor in enumerate(second_part.inputs):
    # Extract the shape and replace any dynamic dimensions (None) with 1 (batch size)
    static_shape = [1 if dim is None else dim for dim in tensor.shape]
    
    static_input = tf.keras.Input(
        batch_shape=static_shape, 
        dtype=tensor.dtype, 
        name=f"static_second_input_{i}"
    )
    static_inputs_second_part.append(static_input)

static_second_part = tf.keras.Model(
    inputs=static_inputs_second_part,
    outputs=second_part(static_inputs_second_part)
)

converter2 = tf.lite.TFLiteConverter.from_keras_model(static_second_part)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.target_spec.supported_types = [tf.int8]
converter2.inference_input_type = tf.int8
converter2.inference_output_type = tf.int8

converter2.representative_dataset = representative_data_gen_second
second_chunk_qint8_tflite = converter2.convert()

with open('/tmp/diffusion_model_second_qint8.tflite', 'wb') as f:
        f.write(second_chunk_qint8_tflite)

INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp7_m5rurx\assets


INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp7_m5rurx\assets
d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow\lite\python\convert.py:789: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
